## 3.2 PDE有限差分：热传导与CFL条件（2学时）

**本节目标：** 能把方程、初边值条件和显式时间步长限制一起离散。

### 3.2.1 概念讲解：把问题说明白

设一根长度为 $L$ 的细长均匀导热杆，其温度为 $u(x,t)$，其中 $x$ 为沿杆长的空间坐标（m），$t$ 为时间（s）。假定材料均匀、热扩散系数 $\kappa>0$ 为常数、无内热源且忽略横向温度变化，则温度满足

$$
\frac{\partial u}{\partial t}
=\kappa\frac{\partial^2u}{\partial x^2},
\qquad 0<x<L,\quad t>0.
$$

这里 $[\kappa]=\mathrm{m^2/s}$。方程右侧的空间二阶导数刻画温度分布的局部弯曲程度：局部温度峰值通常逐渐下降，谷值通常逐渐上升，因此热方程具有平滑作用。

方程本身不能单独确定唯一的温度演化，还要给出初始温度及两端的边界条件。本节采用**零 Dirichlet 边界**：

$$
u(0,t)=u(L,t)=0,\qquad u(x,0)=u_0(x).
$$

在要求经典解保持足够光滑时，初值应满足角点相容性 $u_0(0)=u_0(L)=0$。边界处理属于初边值问题的一部分，不能只对内部节点离散而忽略端点条件。

作为贯穿本节的教学算例，取

$$
L=1\,\mathrm{m},\qquad
\kappa=0.1\,\mathrm{m^2/s},\qquad
A=1\,^\circ\mathrm{C},
$$

$$
u(x,0)=A\sin\!\left(\frac{\pi x}{L}\right),\quad
u(0,t)=u(L,t)=0.
$$

此时连续方程的**独立解析解**为

$$
\boxed{
u_{\mathrm{ref}}(x,t)
=A\sin\!\left(\frac{\pi x}{L}\right)
\exp\!\left[-\kappa\left(\frac{\pi}{L}\right)^2t\right].
}
$$

这个解析解由连续热方程直接得到，而不是由 FTCS 程序计算出来。后面的网格收敛实验使用它作为参考。

为了验证违反 CFL 条件时高频误差的增长，稳定性对照实验还在同一 $J=40$ 网格的初始温度中加入幅值极小、与零边界相容的高频模态：

$$
u_\varepsilon(x,0;J)
=A\sin\!\left(\frac{\pi x}{L}\right)
+\varepsilon\sin\!\left(\frac{(J-1)\pi x}{L}\right),
\qquad\varepsilon=10^{-12}\,^\circ\mathrm{C}.
$$

因此这一对照实验必须采用包含**两个正弦模态**的解析解：

$$
\begin{aligned}
u_{\varepsilon,\mathrm{ref}}(x,t;J)
={}&A\sin\!\left(\frac{\pi x}{L}\right)
e^{-\kappa(\pi/L)^2t}\\
&+\varepsilon\sin\!\left(\frac{(J-1)\pi x}{L}\right)
e^{-\kappa((J-1)\pi/L)^2t}.
\end{aligned}
$$

注意：**网格加密实验不能保留随 $J$ 改变频率的人为高频扰动**，否则不同网格求解的不是同一个连续初值问题。

### 关键推导：让结论有依据

**均匀时空网格。** 将区间 $[0,L]$ 均分为 $J$ 个小区间，取

$$
x_j=j\Delta x,\qquad \Delta x=\frac{L}{J},
\quad j=0,\ldots,J;
\qquad t_n=n\Delta t.
$$

用大写 $U_j^n$ 表示精确温度 $u(x_j,t_n)$ 的离散近似。初始层与边界直接赋值为

$$
U_j^0=u_0(x_j),\qquad U_0^n=U_J^n=0.
$$

只需对内部节点 $j=1,\ldots,J-1$ 递推。

**空间二阶中心差分。** 固定 $t=t_n$，分别在 $x_j\pm\Delta x$ 作 Taylor 展开（以下各导数取于 $(x_j,t_n)$）：

$$
\begin{aligned}
u(x_j+\Delta x,t_n)
&=u+\Delta x\,u_x+\frac{(\Delta x)^2}{2}u_{xx}
+\frac{(\Delta x)^3}{6}u_{xxx}
+\frac{(\Delta x)^4}{24}u_{xxxx}
+\mathcal{O}((\Delta x)^5),\\
u(x_j-\Delta x,t_n)
&=u-\Delta x\,u_x+\frac{(\Delta x)^2}{2}u_{xx}
-\frac{(\Delta x)^3}{6}u_{xxx}
+\frac{(\Delta x)^4}{24}u_{xxxx}
+\mathcal{O}((\Delta x)^5).
\end{aligned}
$$

相加可消去奇数阶项；在精确解足够光滑的条件下，继续保留对称余项，得到

$$
\frac{u(x_{j+1},t_n)-2u(x_j,t_n)+u(x_{j-1},t_n)}
{(\Delta x)^2}
=u_{xx}+\frac{(\Delta x)^2}{12}u_{xxxx}
+\mathcal{O}((\Delta x)^4).
$$

所以空间中心差分对 $u_{xx}$ 具有二阶精度。

**时间前向 Euler 差分。** 固定 $x=x_j$，沿时间方向展开：

$$
u(x_j,t_{n+1})=u(x_j,t_n)+\Delta t\,u_t
+\frac{(\Delta t)^2}{2}u_{tt}
+\mathcal{O}((\Delta t)^3),
$$

故

$$
\frac{u(x_j,t_{n+1})-u(x_j,t_n)}{\Delta t}
=u_t+\frac{\Delta t}{2}u_{tt}
+\mathcal{O}((\Delta t)^2).
$$

时间前向差分只有一阶精度。用上述时间差分和空间差分近似热方程，得到 FTCS（Forward-Time, Centered-Space）显式格式

$$
\frac{U_j^{n+1}-U_j^n}{\Delta t}
=\kappa\frac{U_{j+1}^n-2U_j^n+U_{j-1}^n}{(\Delta x)^2}.
$$

定义无量纲扩散参数（网格 Fourier 数）

$$
\boxed{r=\frac{\kappa\Delta t}{(\Delta x)^2}},
$$

即可写成直接递推式

$$
\boxed{
U_j^{n+1}=rU_{j-1}^n+(1-2r)U_j^n+rU_{j+1}^n.
}
$$

之所以称为显式格式，是因为下一层的 $U_j^{n+1}$ 只依赖于上一层已经算出的三个节点值。

**局部截断误差与相容性。** 将精确解代入 FTCS 离散方程，定义

$$
\tau_j^n=
\frac{u(x_j,t_{n+1})-u(x_j,t_n)}{\Delta t}
-\kappa
\frac{u(x_{j+1},t_n)-2u(x_j,t_n)+u(x_{j-1},t_n)}
{(\Delta x)^2}.
$$

分别代入前面的 Taylor 展开，得到

$$
\begin{aligned}
\tau_j^n
&=(u_t-\kappa u_{xx})
+\frac{\Delta t}{2}u_{tt}
-\frac{\kappa(\Delta x)^2}{12}u_{xxxx}
+\mathcal{O}((\Delta t)^2+(\Delta x)^4)\\
&=\frac{\Delta t}{2}u_{tt}
-\frac{\kappa(\Delta x)^2}{12}u_{xxxx}
+\mathcal{O}((\Delta t)^2+(\Delta x)^4).
\end{aligned}
$$

因为 $\kappa$ 为常数，且 $u$ 足够光滑，

$$
u_{tt}=\kappa(u_t)_{xx}=\kappa^2u_{xxxx}.
$$

因此

$$
\boxed{
\tau_j^n=
\left(\frac{\kappa^2\Delta t}{2}
-\frac{\kappa(\Delta x)^2}{12}\right)u_{xxxx}
+\mathcal{O}((\Delta t)^2+(\Delta x)^4)
=\mathcal{O}(\Delta t+(\Delta x)^2).
}
$$

这证明 FTCS 与热方程相容，具有**时间一阶、空间二阶**的局部精度；但局部相容并不单独保证多步递推的稳定性。

**CFL 条件的定理与证明。** 本节对应教材 3.2.2.7 中的稳定性论证，不引入未经核实的外部定理来源。

**定理（FTCS 的离散最大值原理及最大范数稳定性）。** 对一维常系数热方程、均匀空间网格和零 Dirichlet 边界，在 FTCS 格式中若

$$
\boxed{0\le r\le\frac12},
$$

则每一步的离散解满足

$$
\|U^{n+1}\|_\infty\le\|U^n\|_\infty,
\qquad \|U^n\|_\infty=\max_{0\le j\le J}|U_j^n|.
$$

**证明。** 当 $0\le r\le1/2$ 时，三个递推系数 $r$、$1-2r$、$r$ 均非负且和为 $1$。因此对任意内部节点，

$$
\begin{aligned}
|U_j^{n+1}|
&\le r|U_{j-1}^n|+(1-2r)|U_j^n|+r|U_{j+1}^n|\\
&\le[r+(1-2r)+r]\|U^n\|_\infty
=\|U^n\|_\infty.
\end{aligned}
$$

边界节点保持零值，对所有节点取最大值得结论；若将 $|U|$ 换为相邻节点的最小值和最大值，也得到相应的离散最大值原理。证毕。

因此本格式标准扩散型 CFL 条件可以写为

$$
\boxed{
\frac{\kappa\Delta t}{(\Delta x)^2}\le\frac12
\quad\Longleftrightarrow\quad
\Delta t\le\frac{(\Delta x)^2}{2\kappa}.
}
$$

**Fourier 模态解释。** 对齐次误差递推，设 $e_j^n=G^n e^{\mathrm{i}j\theta}$，代入得

$$
\begin{aligned}
G(\theta)
&=1+r(e^{\mathrm{i}\theta}-2+e^{-\mathrm{i}\theta})\\
&=1-2r(1-\cos\theta)
=\boxed{1-4r\sin^2(\theta/2)}.
\end{aligned}
$$

对均匀无限网格，所有波数都满足 $|G(\theta)|\le1$ 的条件正是 $0\le r\le1/2$。当 $r>1/2$ 时，接近最高网格频率的某些误差模态满足 $|G|>1$，并可能在时间推进时交替变号、持续增长。对固定 $J$ 且零 Dirichlet 边界的有限网格，更精细的正弦模态界为

$$
r\le\frac{1}{2\cos^2(\pi/(2J))},
$$

它略高于 $1/2$，但当 $J$ 增大时趋于 $1/2$；本节使用与网格数无关且能保证最大值原理的标准条件。

**稳定性与收敛性。** 设 $e_j^n=u(x_j,t_n)-U_j^n$，误差满足

$$
e_j^{n+1}
=r e_{j-1}^n+(1-2r)e_j^n+r e_{j+1}^n
+\Delta t\,\tau_j^n.
$$

在零边界、精确采样初值和标准 CFL 条件下，逐步取最大范数并累加，可得固定终止时间 $T$ 内

$$
\|e^n\|_\infty\le T\max_{j,n}|\tau_j^n|
\le C_T(\Delta t+(\Delta x)^2).
$$

所以**稳定不等于精确**：稳定性保证误差传播受控，而实际误差大小仍取决于网格与时间步长。保持 $r$ 不变并令 $\Delta x$ 减半时，应同步令 $\Delta t$ 缩至原来的 $1/4$，此时可观测到近似二阶的联合网格收敛。

### 实验设计：先预测，再运行

本节采用教材 3.2.3 的同一模型，固定终止时间 $T=0.6\,\mathrm{s}$，以已知解析解为**独立参考**。先比较满足或违反标准 CFL 条件的两组时间步长，再在稳定步长下进行网格加密。

| 实验要素 | 本组具体方案 |
| --- | --- |
| 要回答的问题与运行前预测 | ① $r=0.4$ 时，数值温度应保持有界、误差较小；② $r=0.6$ 时，小幅高频扰动会被放大，后期出现交替振荡；③ 固定 $r=0.4$ 将 $\Delta x$ 减半时，终点误差预计约降为原来的 $1/4$。 |
| 输入、参数、单位、数据来源与划分 | $L=1\,\mathrm m$，$\kappa=0.1\,\mathrm{m^2/s}$，$A=1\,^\circ\mathrm C$，$T=0.6\,\mathrm s$；零 Dirichlet 边界。稳定性对比取 $J=40$、$\varepsilon=10^{-12}\,^\circ\mathrm C$；收敛实验取 $J=20,40,80,160$ 且 $\varepsilon=0$。均为确定性教学算例，没有训练/测试集划分。 |
| 独立参考解/基线及选择理由 | 稳定性对照使用两正弦模态的 $u_{\varepsilon,\mathrm{ref}}$；收敛实验使用单一正弦模态的 $u_{\mathrm{ref}}$。两者均由连续热方程直接求出，不依赖 FTCS 算法输出。 |
| 误差指标、容差及其依据 | 最大节点误差 $E_\infty^n$、内部节点均方根误差 $E_{\mathrm{RMS}}^n$、温度最大模值 $M^n$、最高频正弦模态系数；验收约定：稳定组终点 $E_\infty<5\times10^{-4}\,^\circ\mathrm C$，不稳定组 $E_\infty>10^6\,^\circ\mathrm C$，加密观测阶 $1.8\le p\le2.2$。仅适用于本算例的教学验收。 |
| 环境、种子、设备与运行预算 | 本次独立运行：Python 3.13.5，NumPy 2.3.5，Matplotlib 3.10.8；CPU、NumPy默认 `float64`。无随机抽样，不需要设随机种子。最细网格 $161$ 个节点、$3840$ 个时间步，实测纯数值推进约 0.1638 s（不含绘图，受环境影响）。 |

### 3.2 输入与独立参考（对应 Notebook 代码单元）


In [1]:
# 3.2 输入、参数与独立解析参考
from pathlib import Path
import csv
import sys
import time
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

L = 1.0                # m
kappa = 0.1            # m^2/s
A = 1.0                # deg C，初始温度峰值
T = 0.6                # s
epsilon = 1e-12       # deg C，固定网格稳定性实验的高频微扰
fig_dir = Path("图片")
data_dir = Path("3.2_实验输出")
fig_dir.mkdir(exist_ok=True)
data_dir.mkdir(exist_ok=True)

def u_exact(x, t, J, eps=0.0):
    """由连续热方程求得的独立解析解；eps=0 时只有低频主模态。"""
    u = A * np.sin(np.pi*x/L) * np.exp(-kappa*(np.pi/L)**2*t)
    u += eps * np.sin((J-1)*np.pi*x/L) * np.exp(
        -kappa*((J-1)*np.pi/L)**2*t
    )
    u[0] = u[-1] = 0.0
    return u

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__, "Matplotlib:", matplotlib.__version__)
print("Model: L=1 m, kappa=0.1 m^2/s, T=0.6 s, zero Dirichlet boundary")
print("No random sampling; random seed not required.")


Python: 3.12.10
NumPy: 1.26.4 Matplotlib: 3.9.2
Model: L=1 m, kappa=0.1 m^2/s, T=0.6 s, zero Dirichlet boundary
No random sampling; random seed not required.


### 3.2.2 方法实现与运行

在 $J=40$ 时，$\Delta x=1/40=0.025\,\mathrm m$。标准 CFL 上界为

$$
\Delta t_{\max}=\frac{0.025^2}{2(0.1)}=0.003125\,\mathrm s.
$$

本节取两组时间步：

| 组别 | $r$ | $\Delta x$（m） | $\Delta t$（s） | 更新步数 | 预测 |
| --- | ---: | ---: | ---: | ---: | --- |
| A（稳定） | 0.4 | 0.025 | 0.002500 | 240 | 高频误差衰减，接近解析解 |
| B（不稳定） | 0.6 | 0.025 | 0.003750 | 160 | 高频误差放大，后期出现明显振荡 |

代码对每个内部节点逐层更新，且**每个时间层都显式设置** $U_0^n=U_J^n=0$。稳定与不稳定两组使用相同的初温与终止时间；网格加密实验不加入高频扰动。

In [2]:
# 3.2 FTCS 方法实现、运行与数值记录
def solve_ftcs(J, r, eps=0.0):
    dx = L/J
    dt = r*dx**2/kappa
    steps = int(round(T/dt))
    if not np.isclose(steps*dt, T, rtol=0, atol=1e-12):
        raise ValueError("终止时间 T 必须为 Δt 的整数倍。")
    x = np.linspace(0.0, L, J+1)
    U = u_exact(x, 0.0, J, eps).copy()

    phi = np.sin((J-1)*np.pi*x/L)
    phi[0] = phi[-1] = 0.0
    denominator = float(np.dot(phi[1:-1], phi[1:-1]))
    ts, max_errs, rms_errs, max_temps, high_coeffs = [], [], [], [], []
    for n in range(steps+1):
        t = n*dt
        ref = u_exact(x, t, J, eps)
        delta = U[1:-1] - ref[1:-1]
        ts.append(t)
        max_errs.append(float(np.max(np.abs(U-ref))))
        rms_errs.append(float(np.sqrt(np.mean(delta**2))))
        max_temps.append(float(np.max(np.abs(U))))
        high_coeffs.append(float(
            np.dot(U[1:-1], phi[1:-1])/denominator
        ))
        if n < steps:
            V = U.copy()
            V[1:-1] = U[1:-1] + r*(U[2:] - 2*U[1:-1] + U[:-2])
            V[0] = V[-1] = 0.0
            U = V

    return dict(
        J=J, r=r, dx=dx, dt=dt, N=steps, x=x, U=U,
        reference=u_exact(x, T, J, eps),
        times=np.asarray(ts), E_inf_hist=np.asarray(max_errs),
        E_rms_hist=np.asarray(rms_errs),
        M_hist=np.asarray(max_temps),
        high_coeff=np.asarray(high_coeffs),
        E_inf=max_errs[-1], E_rms=rms_errs[-1]
    )

clock_start = time.perf_counter()
stable = solve_ftcs(J=40, r=0.4, eps=epsilon)
unstable = solve_ftcs(J=40, r=0.6, eps=epsilon)
refinements = [solve_ftcs(J, r=0.4) for J in (20, 40, 80, 160)]
elapsed = time.perf_counter()-clock_start

for name, d in [("A/stable", stable), ("B/unstable", unstable)]:
    print(f"{name}: r={d['r']:.1f}, dx={d['dx']:.5f} m, "
          f"dt={d['dt']:.8f} s, N={d['N']}, "
          f"E_inf={d['E_inf']:.8e} deg C, "
          f"E_rms={d['E_rms']:.8e} deg C, "
          f"M(T)={d['M_hist'][-1]:.8e} deg C")

print("Refinement: J, dx(m), dt(s), N, E_inf(deg C), order")
previous_error = None
for d in refinements:
    order = float("nan") if previous_error is None else np.log2(previous_error/d["E_inf"])
    print(f"{d['J']:3d}, {d['dx']:.8f}, {d['dt']:.9f}, {d['N']:4d}, "
          f"{d['E_inf']:.8e}, {order:.5f}")
    previous_error = d["E_inf"]
print(f"Numerical runtime without plotting: {elapsed:.4f} s")

# 记录每个时间层的误差与高频模态系数。
with (data_dir/"A2组-3.2_timeseries.csv").open("w", encoding="utf-8", newline="") as f:
    writer=csv.writer(f)
    writer.writerow(["case","time_s","E_inf_degC","E_rms_degC",
                     "max_abs_temp_degC","high_mode_coeff_degC"])
    for name, d in [("stable_r0.4",stable),("unstable_r0.6",unstable)]:
        for values in zip(d["times"],d["E_inf_hist"],d["E_rms_hist"],
                          d["M_hist"],d["high_coeff"]):
            writer.writerow([name,*values])

# 记录不同网格上的误差及经验收敛阶。
with (data_dir/"A2组-3.2_refinement.csv").open("w", encoding="utf-8", newline="") as f:
    writer=csv.writer(f)
    writer.writerow(["J","dx_m","dt_s","steps","E_inf_degC","E_rms_degC","order"])
    previous_error=None
    for d in refinements:
        order="" if previous_error is None else np.log2(previous_error/d["E_inf"])
        writer.writerow([d["J"],d["dx"],d["dt"],d["N"],
                         d["E_inf"],d["E_rms"],order])
        previous_error=d["E_inf"]


A/stable: r=0.4, dx=0.02500 m, dt=0.00250000 s, N=240, E_inf=2.35955149e-04 deg C, E_rms=1.68970992e-04 deg C, M(T)=5.52886279e-01 deg C
B/unstable: r=0.6, dx=0.02500 m, dt=0.00375000 s, N=160, E_inf=1.57264043e+11 deg C, E_rms=1.12619204e+11 deg C, M(T)=1.57264043e+11 deg C
Refinement: J, dx(m), dt(s), N, E_inf(deg C), order
 20, 0.05000000, 0.010000000,   60, 9.46641349e-04, nan
 40, 0.02500000, 0.002500000,  240, 2.35955149e-04, 2.00431
 80, 0.01250000, 0.000625000,  960, 5.89449108e-05, 2.00107
160, 0.00625000, 0.000156250, 3840, 1.47334885e-05, 2.00027
Numerical runtime without plotting: 0.4233 s


### 3.2.3 误差图表与比较

误差指标定义为

$$
E_\infty^n=\max_{0\le j\le J}|U_j^n-u_{\mathrm{exact}}(x_j,t_n)|,
$$

$$
E_{\mathrm{RMS}}^n
=\sqrt{\frac{1}{J-1}\sum_{j=1}^{J-1}
[U_j^n-u_{\mathrm{exact}}(x_j,t_n)]^2},
\qquad
M^n=\max_j|U_j^n|.
$$

四张图采用英文坐标标签以便在不同运行环境显示；题注提供中文解释。对数纵轴绘图时以 $10^{-16}$ 为**显示下限**，避免零误差无法显示；记录的 CSV 与表格仍保留实际数值。

In [3]:
# 3.2 误差图表：纵轴与单位、图例和参数均显式注明
# 图 3.2-6：最大节点误差随时间的变化。
fig, ax = plt.subplots(figsize=(8, 4.5))
for label, d in [("Stable: r=0.4",stable),("Unstable: r=0.6",unstable)]:
    ax.semilogy(d["times"], np.maximum(d["E_inf_hist"], 1e-16), label=label)
ax.set(xlabel="Time t (s)",ylabel="Maximum nodal error (deg C)",
       title="FTCS error growth: J=40, kappa=0.1, T=0.6 s")
ax.grid(True,which="both",alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir/"A2组-fig_3_2_3_error.png",dpi=160)
plt.close(fig)

# 图 3.2-7：同一终止时间下，稳定组/不稳定组分别与解析解比较。
fig,axes=plt.subplots(1,2,figsize=(12,4.7),sharex=True)
for ax, name, d in zip(axes,("Stable: r=0.4","Unstable: r=0.6"),(stable,unstable)):
    ax.plot(d["x"], d["reference"], "--", label="Exact")
    ax.plot(d["x"], d["U"], "o-", ms=3, label="FTCS")
    ax.set(xlabel="Position x (m)", ylabel="Temperature (deg C)",
           title=f"{name}; dt={d['dt']:.6f} s")
    ax.grid(True,alpha=0.3)
    ax.legend()
fig.suptitle("Spatial profiles: J=40, kappa=0.1, T=0.6 s (independent y-scales)")
fig.tight_layout()
fig.savefig(fig_dir/"A2组-fig_3_2_3_unstable_profile.png",dpi=160)
plt.close(fig)

# 图 3.2-8：网格最高阶正弦模态系数随时间的变化。
fig,ax=plt.subplots(figsize=(8,4.5))
for name,d in [("Stable: r=0.4",stable),("Unstable: r=0.6",unstable)]:
    ax.semilogy(d["times"],np.maximum(np.abs(d["high_coeff"]),1e-16),label=name)
ax.set(xlabel="Time t (s)",ylabel="Absolute high-mode coefficient (deg C)",
       title="High-mode amplitude: J=40, kappa=0.1")
ax.grid(True,which="both",alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir/"A2组-fig_3_2_3_high_mode.png",dpi=160)
plt.close(fig)

# 图 3.2-9：固定 r=0.4 的双对数网格收敛图。
dxs=np.asarray([d["dx"] for d in refinements])
errors=np.asarray([d["E_inf"] for d in refinements])
fig,ax=plt.subplots(figsize=(8,4.5))
ax.loglog(dxs,errors,"o-",label="Measured E_inf")
ax.loglog(dxs,errors[0]*(dxs/dxs[0])**2,"--",label="Reference O(dx^2)")
ax.set(xlabel="Spatial step dx (m)",ylabel="Max error at T=0.6 s (deg C)",
       title="FTCS refinement: r=0.4, kappa=0.1")
ax.grid(True,which="both",alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(fig_dir/"A2组-fig_3_2_3_convergence.png",dpi=160)
plt.close(fig)

print("Saved 4 figures in:", fig_dir.resolve())

Saved 4 figures in: C:\Users\杜沁霖\course-work\course-2026\教材\第03章-离散的世界\工作\图片
